In [1]:
import numpy as np
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit.circuit.library import IntegerComparator
from math import ceil, log2

from qiskit_aer import AerSimulator
from qiskit.quantum_info import Statevector

In [2]:
def qft_on(circ: QuantumCircuit, qubits):
    n = len(qubits)
    for j in range(n):
        qj = qubits[j]
        circ.h(qj)
        for k in range(j + 1, n):
            qk = qubits[k]
            angle = np.pi / (2 ** (k - j))
            circ.cp(angle, qk, qj)
    # bit reversal
    for j in range(n // 2):
        circ.swap(qubits[j], qubits[n - 1 - j])

In [3]:
def iqft_on(circ: QuantumCircuit, qubits):
    n = len(qubits)
    # undo bit-reversal first
    for j in range(n // 2):
        circ.swap(qubits[j], qubits[n - 1 - j])

    # inverse rotations
    for j in reversed(range(n)):
        qj = qubits[j]
        for k in reversed(range(j + 1, n)):
            qk = qubits[k]
            angle = -np.pi / (2 ** (k - j))  # negative angle
            circ.cp(angle, qk, qj)
        circ.h(qj)

In [4]:
def add_classical_constant_on(circ: QuantumCircuit, qubits, a: int):
    """
    |x> → |x + a (mod 2^n)>
    qubits: MSB → LSB
    """
    qft_on(circ, qubits)
    for j, qj in enumerate(qubits):
        theta = +2 * np.pi * a / (2 ** (j + 1))
        circ.p(theta, qj)
    iqft_on(circ, qubits)

In [5]:
def sub_classical_constant_on(circ: QuantumCircuit, qubits, a: int):
    """
    |x> → |x - a (mod 2^n)>
    """
    qft_on(circ, qubits)
    for j, qj in enumerate(qubits):
        theta = -2 * np.pi * a / (2 ** (j + 1))
        circ.p(theta, qj)
    iqft_on(circ, qubits)

In [6]:
def init_classical_on(circ: QuantumCircuit, reg, value: int):
    """
    reg[0] = MSB, reg[-1] = LSB
    value를 이 순서에 맞게 올려줌.
    """
    n = len(reg)
    bits = format(value, f"0{n}b")  # MSB ... LSB
    for j, b in enumerate(bits):
        if b == "1":
            circ.x(reg[j])

In [7]:
def controlled_sub_classical_on(circ: QuantumCircuit, control, qubits, a: int):
    """
    control=1 일 때만 |x> -> |x - a (mod 2^n)>
    qubits: MSB → LSB
    """
    # 1) QFT
    qft_on(circ, qubits)

    # 2) control이 1일 때만 phase 적용
    for j, qj in enumerate(qubits):
        theta = -2 * np.pi * a / (2 ** (j + 1))
        circ.cp(theta, control, qj)

    # 3) iQFT
    iqft_on(circ, qubits)

In [8]:
def build_capacity_check_unitary(weights, capacity):
    """
    QTG 스타일로 capacity에서 w를 빼 가면서,
    각 아이템 위치에서 '용량 위반 여부'를 cond[m]에 기록하는 unitary.
    
    레지스터 순서 (위→아래, forward=True 기준):
      path[0..n-1] | cap[0..n_cap-1] | cond[0..n-1] | ge[0] | wcmp[...] | ctrl[0]
    """

    n = len(weights)
    n_cap = max(1, ceil(log2(capacity + 1)))

    # 1) dummy comparator로 ancilla 개수 계산
    dummy_cmp = IntegerComparator(num_state_qubits=n_cap,
                                  value=0,
                                  geq=True)
    total_cmp_qubits = dummy_cmp.num_qubits       # = n_cap + 1(flag) + n_anc
    n_anc = total_cmp_qubits - (n_cap + 1)        # ancilla 수

    # 2) 레지스터 정의
    path = QuantumRegister(n,      "path")
    cap  = QuantumRegister(n_cap,  "cap")
    cond = QuantumRegister(n,      "cond")
    ge   = QuantumRegister(1,      "ge")

    if n_anc > 0:
        wcmp = QuantumRegister(n_anc, "wcmp")
        ctrl = QuantumRegister(1,    "ctrl")
        qc = QuantumCircuit(path, cap, cond, ge, wcmp, ctrl)
    else:
        wcmp = None
        ctrl = QuantumRegister(1, "ctrl")
        qc = QuantumCircuit(path, cap, cond, ge, ctrl)

    # 3) path H (테스트용)
    for q in path:
        qc.h(q)

    # 4) capacity = C 로드 (MSB→LSB)
    init_classical_on(qc, cap, capacity)

    # 5) 각 아이템 m에 대해: 비교 + 조건부 SUB + 위반 표시
    for m, w_m in enumerate(weights):

        # (a) comparator 서브서킷: cap >= w_m ?
        cmp_circ = IntegerComparator(num_state_qubits=n_cap,
                                     value=w_m,
                                     geq=True)

        if n_anc > 0:
            cmp_qubits = list(cap) + [ge[0]] + list(wcmp)
        else:
            cmp_qubits = list(cap) + [ge[0]]

        # → 'cmp'라는 게이트로 append하는 게 아니라,
        #    서브회로를 통째로 compose해서 회로에 풀어 넣음
        qc.compose(cmp_circ, cmp_qubits, inplace=True)

        # (b) ctrl = path[m] ∧ ge
        qc.ccx(path[m], ge[0], ctrl[0])

        # (c) ctrl=1이면 cap -= w_m
        controlled_sub_classical_on(qc, ctrl[0], cap, w_m)

        # (d) cond[m] = path[m] ∧ (NOT ge)  → 위반 표시
        qc.x(ge[0])
        qc.ccx(path[m], ge[0], cond[m])
        qc.x(ge[0])

        # (e) ctrl 다시 0으로 복원
        qc.ccx(path[m], ge[0], ctrl[0])

        # (f) comparator uncompute (inverse 서브회로를 다시 compose)
        qc.compose(cmp_circ.inverse(), cmp_qubits, inplace=True)

    return qc

In [9]:
def print_full_statevector_clean(sv, threshold=1e-6, forward=False):
    """
    범용 Statevector 출력 함수.
    - forward=False  : MSB->LSB (사람 기준, Qiskit 기준)
    - forward=True   : LSB->MSB (QFT/ADD/SUB 디버깅용)
                      + index도 LSB 기준 숫자 증가 순으로 정렬
    
    """
    if not isinstance(sv, Statevector):
        sv = Statevector(sv)
    
    amps = sv.data
    n = sv.num_qubits

    print(f"Full Cleaned Statevector (threshold={threshold}, forward={forward})")
    print("-" * 60)

    # 1) 인덱스 리스트 준비
    indices = list(range(len(amps)))

    if forward:
        # ★ forward=True 일 때는 index를 비트 뒤집은 기준으로 정렬해야 함
        def bit_reverse(i):
            raw = format(i, f'0{n}b')  # q2 q1 q0
            rev = raw[::-1]           # q0 q1 q2
            return int(rev, 2)        # rev를 다시 숫자로 해석

        indices.sort(key=lambda i: bit_reverse(i))
    else:
        # 기본은 기존 인덱스 순서 그대로 출력
        pass

    # 2) 출력
    for i in indices:
        amp = amps[i]
        raw = format(i, f'0{n}b')      # q2 q1 q0

        if forward:
            bitstring = raw[::-1]      # LSB->MSB
        else:
            bitstring = raw            # MSB->LSB

        if abs(amp) < threshold:
            # print(f"|{bitstring}> : 0")
            pass
        else:
            print(f"|{bitstring}> : {amp.real:+.6f}{amp.imag:+.6f}j")

    print("-" * 60)

In [10]:
weights = [5, 5, 5]
capacity = 5

qc = build_capacity_check_unitary(weights, capacity)
print(qc.draw())  # 회로 구조 한 번 눈으로 보기

        ┌───┐                                                               »
path_0: ┤ H ├──────────■────────────────────────────────■───────────────────»
        ├───┤          │                                │                   »
path_1: ┤ H ├──────────┼────────────────────────────────┼───────────────────»
        ├───┤          │                                │                   »
path_2: ┤ H ├──────────┼────────────────────────────────┼───────────────────»
        ├───┤┌──────┐  │  ┌───┐                         │                   »
 cap_0: ┤ X ├┤0     ├──┼──┤ H ├─■────────■──────────────┼─────────────────X─»
        └───┘│      │  │  └───┘ │P(π/2)  │       ┌───┐  │                 │ »
 cap_1: ─────┤1     ├──┼────────■────────┼───────┤ H ├──┼───■─────────────┼─»
        ┌───┐│      │  │                 │P(π/4) └───┘  │   │P(π/2) ┌───┐ │ »
 cap_2: ┤ X ├┤2     ├──┼─────────────────■──────────────┼───■───────┤ H ├─X─»
        └───┘│      │  │                              ┌─┴─┐     

In [11]:
sv = Statevector.from_instruction(qc)
print_full_statevector_clean(sv, threshold=1e-6, forward=True)qc.draw("mpl")

Full Cleaned Statevector (threshold=1e-06, forward=True)
------------------------------------------------------------
|0001010000000> : +0.353553+0.000000j
|0010000001000> : +0.353553+0.000000j
|0100000001000> : +0.353553+0.000000j
|0110110000000> : +0.353553+0.000000j
|1000000001000> : +0.353553+0.000000j
|1010110000000> : +0.353553+0.000000j
|1100110000000> : +0.353553+0.000000j
|1111100001000> : +0.353553+0.000000j
------------------------------------------------------------


In [1]:
qc.draw("mpl")

NameError: name 'qc' is not defined